In [63]:
import tensorflow as tf
import numpy as np
import os


In [64]:
class_name=os.listdir('data')
class_name

['Blight', 'Common_Rust', 'Gray_Leaf_Spot', 'Healthy']

In [65]:
directory="data"

In [66]:
train_ds=tf.keras.utils.image_dataset_from_directory(
    directory,
    subset='training',
    validation_split=0.3,
    seed=42,
    image_size=(128,128),
    batch_size=20

)

val_ds=tf.keras.utils.image_dataset_from_directory(
    directory,
    subset='validation',
    seed=42,
    image_size=(128,128),
    batch_size=20,
    validation_split=0.3

)

Found 4188 files belonging to 4 classes.
Using 2932 files for training.
Found 4188 files belonging to 4 classes.
Using 1256 files for validation.


In [67]:
auto_tune =tf.data.AUTOTUNE
train_ds=train_ds.cache().prefetch(buffer_size=auto_tune)
val_ds =val_ds.cache().prefetch(buffer_size=auto_tune)

In [68]:
base_model = tf.keras.applications.VGG16(
    weights='imagenet',       
    include_top=False,          
    input_shape=(128, 128, 3)   
)
base_model.trainable = False   


In [69]:
from tensorflow.keras.layers import(Flatten,Dense,Dropout,RandomRotation,RandomFlip,Rescaling,Input,Conv2D,MaxPooling2D,RandomZoom)
from tensorflow.keras.callbacks import(EarlyStopping,ModelCheckpoint)
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint

In [70]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy


In [71]:
model=Sequential([
    Input(shape=(128,128,3)),
    RandomFlip('horizontal_and_vertical'),
    RandomZoom(0.5),
    RandomRotation(0.9),
    base_model,

    Rescaling(1./255),
    Conv2D(filters=80,kernel_size=(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Conv2D(filters=120,kernel_size=(3,3),activation='relu',padding='same'),
    Flatten(),
    Dense(200,activation='relu'),
    Dropout(0.3),
    Dense(100,activation='relu'),
    Dense(4,activation='softmax')

])
model.summary()
model.compile(optimizer='adamax',loss=SparseCategoricalCrossentropy(from_logits=False),metrics=['accuracy'])

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_13 (RandomFlip)     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_13              │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_12 (Rescaling)        │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 4, 4, 80)       │       368,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 2, 2, 80)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 2, 2, 120)      │        86,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_13 (Flatten)            │ (None, 480)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 200)            │        96,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 100)            │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 4)              │           404 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,286,632 (58.31 MB)

 Trainable params: 571,944 (2.18 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [72]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[EarlyStopping(patience=3,mode='max',monitor='val_accuracy',restore_best_weights=True),
               ModelCheckpoint('maize_diseases.keras',monitor='val_loss',save_best_only=True,mode='min')

    ]

)

Epoch 1/15


147/147 ━━━━━━━━━━━━━━━━━━━━ 199s 1s/step - accuracy: 0.7060 - loss: 0.6993 - val_accuracy: 0.8185 - val_loss: 0.4016
Epoch 2/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 164s 1s/step - accuracy: 0.8240 - loss: 0.4154 - val_accuracy: 0.8153 - val_loss: 0.4013
Epoch 3/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - accuracy: 0.8291 - loss: 0.3926 - val_accuracy: 0.8511 - val_loss: 0.3455
Epoch 4/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 177s 1s/step - accuracy: 0.8574 - loss: 0.3446 - val_accuracy: 0.8487 - val_loss: 0.3560
Epoch 5/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 308s 2s/step - accuracy: 0.8786 - loss: 0.3030 - val_accuracy: 0.8662 - val_loss: 0.3127
Epoch 6/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 213s 1s/step - accuracy: 0.8772 - loss: 0.2999 - val_accuracy: 0.8734 - val_loss: 0.3117
Epoch 7/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.8915 - loss: 0.2823 - val_accuracy: 0.8654 - val_loss: 0.3153
Epoch 8/15
147/147 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.8909 - loss: 0.2866 - val_accuracy: 0.878

In [73]:
model=tf.keras.models.load_model('maize_diseases.keras')
img=tf.keras.utils.load_img('SAMPLE/health_maize3.jpg',target_size=(128,128))
img_array=tf.keras.utils.img_to_array(img)
norm=img_array/255
batch=np.expand_dims(norm,axis=0)
s=model.predict(batch)
s



1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 618ms/step


array([[0.6708285 , 0.10280526, 0.15623973, 0.07012651]], dtype=float32)